In [1]:
from kafka import KafkaConsumer, TopicPartition
import json
import numpy as np
import matplotlib.mlab as mlab
from sklearn.ensemble import IsolationForest
import os
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from collections import defaultdict
from baskervillehall.baskervillehall_isolation_forest import BaskervillehallIsolationForest
from baskervillehall.model_io import ModelIO

plt.rcParams["figure.figsize"] = (15,5)

In [2]:
kafka_url = ['kafka9-0.kafka9-headless.default.svc.cluster.local:9093','kafka9-1.kafka9-headless.default.svc.cluster.local:9093','kafka9-2.kafka9-headless.default.svc.cluster.local:9093']
topic = 'BASKERVILLEHALL_3'

In [3]:
datetime_format = '%Y-%m-%d %H:%M:%S'
path = 'anton/dataset/'

In [4]:
os.environ['S3_ACCESS'] = 'c490f17bdb784fa08f4d11836ee18e48'
os.environ['S3_SECRET'] = 'e4b65e27b3734d0d96ce6038586ef43f'
os.environ['S3_ENDPOINT'] = 's3.gra.cloud.ovh.net'
os.environ['S3_REGION'] = 'GRA'
s3_connection = {
    's3_access':os.environ['S3_ACCESS'],
    's3_secret':os.environ['S3_SECRET'],
    's3_endpoint': os.environ['S3_ENDPOINT'],
    's3_region':os.environ['S3_REGION']
}

In [22]:
def read_sessions(hosts, size):
    consumer = KafkaConsumer(
        bootstrap_servers=kafka_url,
        # group_id='anton12'
    )
    
    ids = []

    print(f'Reading from kafka. Hosts = {hosts} ...')
    time_now = int(time.time())
    sessions = []

    consumer.assign([
        TopicPartition(topic, 0),
        TopicPartition(topic, 1),
        TopicPartition(topic, 2),
    ])
    consumer.seek_to_beginning()
    complete = False
    host_count = {host: 0 for host in hosts}
    host_complete = {}
    host_sessions = {host: [] for host in hosts}
    first_run = True
    while not complete:
        raw_messages = consumer.poll(timeout_ms=1000, max_records=5000)

        for topic_partition, messages in raw_messages.items():
            for message in messages:
                # prevent from getting messages too close to the current time
                time_diff_in_minutes = (time_now - message.timestamp / 1000) / 60
                if time_diff_in_minutes < 2:
                    print(f'{time_diff_in_minutes} minutes. Topic offset is too close to the current times...')
                    complete = True
                    break
                
                if message.value is None :
                    continue
                if message.key is None:
                    continue
                host = message.key.decode("utf-8")
                if host not in host_count.keys():
                    continue
                if host in host_complete.keys():
                    continue

                value = json.loads(message.value.decode("utf-8"))
                value['duration'] = abs(value['duration'])
                duration = value['duration']
                if duration < 1:
                    continue
                if duration > 300:
                    continue
                if len(value['requests']) < 3:
                    continue
                    
                if first_run:
                    first_run = False
                    print(f'First session start = {value["start"]} end = {value["end"]}')
                    
                host_sessions[host].append(value)
                ids.append(value['ip'])
                host_count[host] += 1
                if host_count[host] == size:
                    host_complete[host] = True
                    if len(host_complete.keys()) == len(hosts):
                        complete = True
                        break
                if host_count[host] % 100 == 0:
                    print(f'{host_count[host]} sessions read', value['end'], host)
            
    return np.array(ids), host_sessions, host_count



In [23]:
ids, host_sessions, host_count = read_sessions([
    'verafiles.org', 
    'zhitomir.info',
    'telegraf.in.ua',
    'kavkaz-uzel.eu',
    'gubernia.com'
], 7000)

Reading from kafka. Hosts = ['verafiles.org', 'zhitomir.info', 'telegraf.in.ua', 'kavkaz-uzel.eu', 'gubernia.com'] ...
First session start = 2023-12-26 08:40:50 end = 2023-12-26 08:41:02
100 sessions read 2023-12-26 08:50:02 kavkaz-uzel.eu
200 sessions read 2023-12-26 08:52:31 kavkaz-uzel.eu
300 sessions read 2023-12-26 08:54:23 kavkaz-uzel.eu
400 sessions read 2023-12-26 08:55:54 kavkaz-uzel.eu
500 sessions read 2023-12-26 08:58:39 kavkaz-uzel.eu
100 sessions read 2023-12-26 12:15:50 zhitomir.info
600 sessions read 2023-12-26 09:02:37 kavkaz-uzel.eu
700 sessions read 2023-12-26 08:59:30 kavkaz-uzel.eu
800 sessions read 2023-12-26 09:08:31 kavkaz-uzel.eu
900 sessions read 2023-12-26 09:11:27 kavkaz-uzel.eu
100 sessions read 2023-12-26 10:52:50 telegraf.in.ua
1000 sessions read 2023-12-26 09:13:25 kavkaz-uzel.eu
1100 sessions read 2023-12-26 09:15:27 kavkaz-uzel.eu
200 sessions read 2023-12-26 12:17:44 zhitomir.info
1200 sessions read 2023-12-26 09:18:07 kavkaz-uzel.eu
1300 sessions rea

In [24]:
host_count

{'verafiles.org': 7000,
 'zhitomir.info': 7000,
 'telegraf.in.ua': 7000,
 'kavkaz-uzel.eu': 7000,
 'gubernia.com': 1994}

In [25]:
model_io = ModelIO(**s3_connection)
index = 1
folder = 'dataset5'
for host, sessions in host_sessions.items():
    subfolder = os.path.join(path, folder, f'{index}')
    model_io._save_object(json.dumps(sessions, indent=2), os.path.join(subfolder, 'sessions.json'))
    model_io._save_object(json.dumps({
        'host': host,
        'count': len(sessions)
    }, indent=2), os.path.join(subfolder, 'info.json'))
    index += 1
    
    